# PaddleOCRv5 Invoice Parser — Demo
Pipeline: Gambar struk → JSON (AI-DS-SPEC)

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.pipeline import build_ocr_engine, parse_invoice

# Ganti path ke gambar struk Anda
IMAGE_PATH = 'tests/samples/sample_receipt.jpg'

engine = build_ocr_engine(
    det_model_dir='../PP-OCRv5_server_det',
    rec_model_dir='../PP-OCRv5_server_rec',
)
result = parse_invoice(IMAGE_PATH, engine)
print(json.dumps(result, ensure_ascii=False, indent=2))

## Visualisasi tiap tahap pipeline

In [ ]:
from paddleocr import PaddleOCR
from src.ocr_adapter import adapt
from src.line_builder import build_lines
from src.field_extractor import extract_merchant, extract_date, extract_total, extract_items
from src.category_mapper import map_category
from src.confidence import compute_confidence

# Step 1: OCR
ocr_raw = engine.ocr(IMAGE_PATH, cls=True)
words = adapt(ocr_raw)
print(f'Words extracted: {len(words)}')

# Step 2: Build lines
lines = build_lines(words)
for i, l in enumerate(lines):
    print(f'Line {i:02d} [y_norm={l["y_norm"]:.2f}] conf={l["avg_conf"]:.2f}: {l["text"]}')

# Step 3: Fields
merchant, m_conf = extract_merchant(lines)
date, d_conf     = extract_date(lines)
total, t_conf    = extract_total(lines)
items, i_conf    = extract_items(lines)

print(f'\nMerchant : {merchant} (conf={m_conf:.2f})')
print(f'Date     : {date} (conf={d_conf:.2f})')
print(f'Total    : {total} (conf={t_conf:.2f})')
print(f'Items    : {items}')
